# Where and what to buy

**Brief: Where (neighbourhoods) + What (property types and mix).**

- Revenue, nightly price and occupancy by neighbourhood and type
- Which segments have the best yield once purchase price is included
- Watch out: segments with few listings (`n`) have unreliable medians

In [37]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
T = config.TABLES   # run `python run.py` from the repo root first if outputs/ is empty

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

## Data
- `seg`: neighbourhood × type × bedrooms (active entire homes)
- `uw`: segments + purchase price, costs, NOI, net yield

### From 01 (keep in mind)
- **Occupancy & revenue** = Inside Airbnb's estimates (`estimated_occupancy_l365d`, `estimated_revenue_l365d`), 12 months to 15 June 2026. Estimates, not real bookings.
- **Price** = one night's quote around June–July 2026 (World Cup period), so revenue may be a bit high.
- **`is_active`** = reviewed in the last 12 months. Only active listings are used for segments.
- Cleaned data: 6,509 → 4,447 listings (no price, price outside $30–$2,000, 90+ night minimum removed: BC/Vancouver short-term = under 90 nights).

In [38]:
seg = pd.read_csv(T / "segments.csv", dtype={"bedrooms_grp": str})
uw = pd.read_csv(T / "segments_underwritten.csv", dtype={"bedrooms_grp": str})

## Quick look

In [39]:
uw[uw["n"] >= 5].sort_values("net_yield", ascending=False).head(20)

,neighbourhood_cleansed,bucket,bedrooms_grp,n,adr_median,occ_median,rev_median,rev_p25,rev_p75,rating,share_commercial,share_licensed_est,side,gvr_type,benchmark_price_cad,price_cad,noi,cash_in,net_yield
97,Strathcona,condo,1,10,406.50,0.46,"83,406.00","30,499.75","89,265.75",4.96,0.00,1.00,East,Apartment,635100,"539,835.00","46,430.63","586,428.40",0.08
14,Downtown Eastside,condo,3+,13,491.00,0.33,"81,345.00","20,790.00","117,612.00",4.67,0.46,0.85,East,Apartment,635100,"952,650.00","43,770.10","1,015,756.00",0.04
83,Riley Park,condo,2,7,353.20,0.31,"65,695.00","26,764.00","77,775.00",4.88,0.00,1.00,East,Apartment,635100,"730,365.00","33,638.45","784,579.60",0.04
12,Downtown Eastside,condo,1,152,368.16,0.35,"49,140.00","17,515.50","75,666.75",4.88,0.14,0.95,East,Apartment,635100,"539,835.00","22,787.10","586,428.40",0.04
6,Downtown,condo,1,374,409.85,0.35,"51,788.00","21,264.75","83,044.50",4.88,0.15,0.95,West,Apartment,751400,"638,690.00","24,317.65","689,237.60",0.04
7,Downtown,condo,2,292,542.12,0.33,"63,091.50","24,832.00","101,596.50",4.88,0.18,0.98,West,Apartment,751400,"864,110.00","31,440.80","923,674.40",0.03
40,Kensington-Cedar Cottage,condo,1,7,275.50,0.44,"44,095.00","40,791.00","49,275.00",4.82,0.14,1.00,East,Apartment,635100,"539,835.00","19,306.04","586,428.40",0.03
8,Downtown,condo,3+,121,575.00,0.33,"75,345.00","28,758.00","125,715.00",4.82,0.38,0.93,West,Apartment,751400,"1,127,100.00","39,106.75","1,197,184.00",0.03
111,West End,condo,2,25,513.53,0.28,"59,953.00","27,510.00","91,837.00",4.91,0.16,0.96,West,Apartment,751400,"864,110.00","29,275.24","923,674.40",0.03
54,Kitsilano,condo,1,33,370.88,0.31,"46,221.00","14,858.00","68,544.00",4.88,0.12,1.00,West,Apartment,751400,"638,690.00","20,476.42","689,237.60",0.03


In [40]:
uw.groupby("bucket")[["rev_median", "price_cad", "net_yield"]].median()

,rev_median,price_cad,net_yield
bucket,,,
condo,"31,797.00","730,365.00",0.01
house,"35,710.50","2,899,700.00",0.01


## Analysis
_Start here._

### Step 1: Demand by neighbourhood
Active entire-home listings only (what DDD would run). Typical (median) value per neighbourhood. Hover a bar to see all the numbers for that neighbourhood.

In [41]:
import altair as alt

df = pd.read_csv(config.PROCESSED / "listings_clean.csv", low_memory=False)
d = df[df["is_active"] & (df["room_type"] == "Entire home/apt")]

hood = (d.groupby("neighbourhood_cleansed")
          .agg(listings=("id", "size"), revenue=("revenue", "median"), total_revenue=("revenue", "sum"),
               occupancy=("occupancy", "median"), price=("price_num", "median"))
          .reset_index())
hood

,neighbourhood_cleansed,listings,revenue,total_revenue,occupancy,price
0,Arbutus Ridge,64,"38,682.00","3,579,934.00",0.30,400.00
1,Downtown,798,"58,486.00","51,770,098.00",0.35,460.97
2,Downtown Eastside,290,"44,043.00","15,640,536.00",0.28,401.18
3,Dunbar Southlands,125,"32,886.00","5,417,985.00",0.18,385.72
4,Fairview,18,"45,239.00","1,114,288.00",0.26,435.75
5,Grandview-Woodland,128,"36,199.50","5,402,280.00",0.25,350.88
6,Hastings-Sunrise,176,"42,266.50","8,842,193.00",0.34,312.88
7,Kensington-Cedar Cottage,217,"40,741.00","10,291,055.00",0.33,350.00
8,Kerrisdale,46,"32,396.50","2,093,233.00",0.20,399.98
9,Killarney,73,"24,960.00","2,864,865.00",0.21,319.53


In [42]:
def bar(col, title, fmt, note=""):
    tip = [alt.Tooltip("neighbourhood_cleansed:N", title="Neighbourhood"),
           alt.Tooltip("revenue:Q", title="Typical yearly revenue", format="$,.0f"),
           alt.Tooltip("occupancy:Q", title="Typical occupancy", format=".0%"),
           alt.Tooltip("price:Q", title="Typical nightly price", format="$,.0f"),
           alt.Tooltip("listings:Q", title="Listings")]
    chart = (alt.Chart(hood)
             .mark_bar(color="#2a78d6", cornerRadiusEnd=4, height=16)
             .encode(x=alt.X(f"{col}:Q", title=None, axis=alt.Axis(format=fmt, grid=True, gridColor="#eee")),
                     y=alt.Y("neighbourhood_cleansed:N", sort="-x", title=None),
                     tooltip=tip)
             .properties(width=550, height=alt.Step(20),
                         title=alt.Title(title, subtitle=note, anchor="start", subtitleColor="#52514e"))
             .configure_view(stroke=None))
    display(chart)

bar("revenue", "Typical yearly revenue per listing (CAD)", "$,.0f",
    note="Yearly revenue = occupancy (nights booked) × nightly price · 12 months to June 2026")
bar("occupancy", "Typical occupancy (share of year booked)", ".0%")
bar("listings", "Number of active whole-home listings", ",d",
    note="Supply: more listings = proven demand, but also more competition")
bar("total_revenue", "Total yearly revenue, all active whole-home listings (CAD)", "$,.0f",
    note="Size of the market = sum of every listing's revenue · 12 months to June 2026")
bar("price", "Typical nightly price (CAD)", "$,.0f",
    note="Quote for one night around June–July 2026 (World Cup period)")

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)